# Lesson 8 — Planning

**You will:** preview what the agent intends to do before letting it act. Decide when plan-then-execute beats act-and-react.

[Open in Colab](https://colab.research.google.com/github/richey-malhotra/barebear/blob/main/lessons/08-planning/lesson.ipynb)
[Read the lesson narrative](./lesson.md)
[Back to syllabus](../README.md)

## The big idea

Two strategies for multi-step tasks:

1. **Act-and-react** — the agent picks a tool, runs it, observes, picks the next. One step at a time. This is what `bear.run()` does.
2. **Plan-then-execute** — the agent first writes down a plain-language plan *without* calling tools. You read the plan. If you like it, you execute. If you don't, you adjust.

Plan-then-execute is slower and less flexible, but it gives you a *preview*. For high-stakes tasks (refactoring code, scheduling meetings, making purchases), the preview is invaluable. For exploratory tasks, it's overhead.

The skill: knowing which to use, when.

## Setup

Run the next two cells once per session. They install barebear and set your OpenRouter key (free tier — get one at <https://openrouter.ai>).

In [ ]:
%pip install -q "barebear[openai]"

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Paste your OpenRouter key (sk-or-...): ")

# Optional: pin a specific free-tier model. If unset, barebear uses its default.
# Free-tier model availability rotates — swap here if a lesson cell errors with
# "model not available". Any *:free model on https://openrouter.ai/models works.
# os.environ["BAREBEAR_MODEL"] = "meta-llama/llama-3.2-3b-instruct:free"

print("Key set. First 8 characters:", os.environ["OPENROUTER_API_KEY"][:8] + "...")
print("Model:", os.environ.get("BAREBEAR_MODEL", "(framework default)"))

## Plan first, then decide

In [ ]:
from barebear import Bear, Task, Tool, OpenRouterModel

def read_file(path: str) -> str:
    return f"# fake contents of {path}\nimport flask\napp = flask.Flask(__name__)"

def edit_file(path: str, change: str) -> str:
    return f"applied: {change}"

bear = Bear(
    model=OpenRouterModel(),
    tools=[
        Tool(name="read_file", fn=read_file, description="Read a source file"),
        Tool(name="edit_file", fn=edit_file, description="Apply a change to a file"),
    ],
)

task = Task(goal="Refactor the auth module to use dependency injection.")

plan = bear.plan(task)
print("--- plan (model wrote, no tools called) ---")
print(plan["plan"])
print()
print(f"plan cost in tokens: {plan['prompt_tokens'] + plan['completion_tokens']}")

## Now decide whether to execute

In a real workflow you'd let a human approve. Here, run the cell to execute.

In [ ]:
report = bear.run(task)
print(report.summary())

## Exercise

1. Plan the same task **three times** with different system prompts (set `Task.system_prompt`). Do the plans differ? Why?
2. Compare token cost: `bear.plan()` vs `bear.run()` for the same task. Which is cheaper, by how much?

## What's next

[Lesson 9 →](../09-reflection/lesson.ipynb)